# Per-Image Pitch Analysis Pipeline

Takes your fine-tuned DINOv2 + Faster R-CNN detector and, for **each image independently**
(no tracking, no cross-image identity — by design, since the goal is generalizing to
unrelated/unseen footage), produces:

- Filtered detections (crowd/stand/advertising-board false positives removed via pitch masking)
- Team assignment for `player`/`goalkeeper` boxes (unsupervised color clustering, per-image)
- Ground position for each detection, normalized to the visible pitch area
- A rough pitch-coverage metric per team

## Explicit limitations (by design, given "discrete images" requirement)
- **Team clustering is per-image only.** "Team A" in one image has no relationship to "Team A"
  in another — there's no continuity without tracking. If an image has too few player detections
  (below `MIN_PLAYERS_FOR_TEAM_SPLIT`), team is reported as `"unknown"` rather than forcing an
  unstable 2-way split.
- **Position is normalized relative to the pitch mask's bounding box, not a true homography.**
  A real broadcast-camera homography needs detected pitch line/corner keypoints, which this
  pipeline doesn't do. This normalization is a reasonable approximation for a rough coverage
  metric, but will distort with strong camera angles/perspective. Treat coverage numbers as
  relative/comparative, not physically precise square-meter figures.
- **Pitch segmentation uses HSV color thresholding**, auto-calibrated per image from a sampled
  center region rather than a fixed constant (to generalize better across stadiums/lighting/turf
  color) — but this can still fail on unusual lighting, artificial turf colors, or heavy shadows.
  Always check the visualization cell below before trusting a batch run.
- **No ball-detection gap-filling** — a missed ball in an image is just absent from that image's
  metadata, consistent with there being no cross-frame interpolation.
- **Goalkeepers are clustered together with players** (k=2) even though a goalkeeper's kit is
  usually a 3rd, distinct color — this is a known simplification, not a bug. If this matters for
  your use case, say so and I can extend it to a 3-cluster version.


In [ ]:
%pip install -q opencv-python-headless scikit-learn --upgrade


In [ ]:
import os
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import cv2
import torch
import torch.nn as nn
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.models.detection.transform import GeneralizedRCNNTransform
from torchvision.ops import MultiScaleRoIAlign
import torchvision.transforms.functional as TF
from sklearn.cluster import KMeans
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
# ============================================================
# CONFIG
# ============================================================
CONFIG = {
    # Path to a checkpoint saved by the training notebook (best_detector.pt or last_checkpoint.pt).
    # Model architecture (backbone size, freeze mode, img_size) is read FROM the checkpoint's
    # saved config — you don't need to re-specify it here.
    # On Kaggle:      "/kaggle/input/<dataset>/best_detector.pt"
    # On your own PC: any local path, e.g. "C:/models/best_detector.pt" or "./best_detector.pt"
    #                 (on Windows, use forward slashes or an r"..." raw string — a plain
    #                 backslash string breaks on escape sequences like 	)
    "checkpoint_path": None,

    # Folder of images to run the pipeline on (not necessarily your training data —
    # this is meant to run on new/unseen images too).
    "images_dir": None,        # e.g. "/kaggle/input/<dataset>/test/images" or a local folder

    # Kaggle's /kaggle/working doesn't exist on your own PC — "./pipeline_output.jsonl" writes
    # to whatever folder you launch the notebook from, and works the same on both.
    "output_jsonl_path": "./pipeline_output.jsonl",

    "detection_score_threshold": 0.5,
    "min_players_for_team_split": 4,   # below this, team is reported as "unknown" for all players
    "num_visualization_samples": 4,    # how many images to show in the sanity-check cell
}

if CONFIG["checkpoint_path"] is None:
    # Auto-detect: search /kaggle/input for a checkpoint if not explicitly set (only relevant on
    # Kaggle — on your own PC this folder won't exist, so just set checkpoint_path manually).
    for dirpath, dirnames, filenames in os.walk("/kaggle/input") if os.path.isdir("/kaggle/input") else []:
        for fn in ("best_detector.pt", "last_checkpoint.pt"):
            if fn in filenames:
                CONFIG["checkpoint_path"] = os.path.join(dirpath, fn)
                break
        if CONFIG["checkpoint_path"]:
            break
    if CONFIG["checkpoint_path"]:
        print("Auto-detected checkpoint:", CONFIG["checkpoint_path"])
    else:
        raise RuntimeError("No checkpoint found under /kaggle/input — set CONFIG['checkpoint_path'] manually.")

if CONFIG["images_dir"] is None:
    raise RuntimeError("Set CONFIG['images_dir'] to a folder of images to run the pipeline on.")


In [ ]:
# ============================================================
# Load the trained model — architecture is reconstructed from the checkpoint's own saved config,
# so this matches whatever backbone_train_mode / model_name / img_size you actually trained with.
# ============================================================
PATCH_SIZE = 14
EMBED_DIMS = {
    "dinov2_vits14": 384, "dinov2_vitb14": 768,
    "dinov2_vitl14": 1024, "dinov2_vitg14": 1536,
}
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


class DinoV2Backbone(nn.Module):
    def __init__(self, model_name, train_mode="frozen", unfreeze_last_n_blocks=0):
        super().__init__()
        # pretrained=False: this notebook always loads YOUR fine-tuned weights via
        # load_state_dict() right after construction, which overwrites the backbone entirely —
        # so downloading Meta's generic pretrained weights here would be wasted bandwidth.
        # torch.hub still needs internet the first time to fetch the DINOv2 architecture CODE
        # (a small GitHub repo, separate from weights); that gets cached locally afterward too.
        self.model = torch.hub.load("facebookresearch/dinov2", model_name, pretrained=False)
        self.embed_dim = EMBED_DIMS[model_name]
        self.out_channels = self.embed_dim
        # At inference we don't need gradients regardless of how it was trained.
        for p in self.model.parameters():
            p.requires_grad = False
        self.model.eval()

    def forward(self, x):
        b, c, h, w = x.shape
        with torch.no_grad():
            feats = self.model.forward_features(x)
        patch_tokens = feats["x_norm_patchtokens"]
        gh, gw = h // PATCH_SIZE, w // PATCH_SIZE
        feat_map = patch_tokens.permute(0, 2, 1).reshape(b, self.embed_dim, gh, gw)
        return {"0": feat_map}


checkpoint = torch.load(CONFIG["checkpoint_path"], map_location=device)
train_config = checkpoint["config"]
class_names = checkpoint["class_names"]
num_classes = len(class_names)
print(f"Loaded checkpoint trained with model={train_config['model_name']}, "
      f"img_size={train_config['img_size']}, classes={class_names}")

backbone = DinoV2Backbone(train_config["model_name"])
anchor_generator = AnchorGenerator(sizes=((16, 32, 64, 128, 256),), aspect_ratios=((0.5, 1.0, 2.0),))
roi_pooler = MultiScaleRoIAlign(featmap_names=["0"], output_size=7, sampling_ratio=2)
model = FasterRCNN(
    backbone, num_classes=num_classes + 1,
    rpn_anchor_generator=anchor_generator, box_roi_pool=roi_pooler,
    min_size=train_config["img_size"], max_size=train_config["img_size"],
    image_mean=IMAGENET_MEAN, image_std=IMAGENET_STD,
)
model.transform = GeneralizedRCNNTransform(
    min_size=train_config["img_size"], max_size=train_config["img_size"],
    image_mean=IMAGENET_MEAN, image_std=IMAGENET_STD, size_divisible=PATCH_SIZE,
)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()
print("Model loaded and ready.")


In [ ]:
# ============================================================
# Pitch segmentation — HSV threshold auto-calibrated per image (not a fixed constant), to
# generalize better across stadiums / turf color / lighting than a hardcoded green range would.
# ============================================================
def segment_pitch(image_bgr):
    """Returns (mask, polygon_points, bounding_rect) for the largest green-field region.
    mask: uint8 binary mask, same size as image. polygon_points: Nx2 array (may be None if no
    pitch found). bounding_rect: (x, y, w, h) of the pitch region (may be None)."""
    h, w = image_bgr.shape[:2]
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)

    # Sample a center-weighted region to estimate this image's actual pitch hue — broadcast
    # footage is usually framed with the pitch dominating the middle of the frame, so this is a
    # reasonable (not perfect) way to auto-calibrate instead of assuming one fixed green shade.
    cy0, cy1 = int(h * 0.3), int(h * 0.85)
    cx0, cx1 = int(w * 0.2), int(w * 0.8)
    sample = hsv[cy0:cy1, cx0:cx1].reshape(-1, 3)
    median_hue = np.median(sample[:, 0])

    lower = np.array([max(0, median_hue - 25), 30, 30], dtype=np.uint8)
    upper = np.array([min(179, median_hue + 25), 255, 255], dtype=np.uint8)
    mask = cv2.inRange(hsv, lower, upper)

    kernel = np.ones((15, 15), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return np.zeros((h, w), dtype=np.uint8), None, None

    largest = max(contours, key=cv2.contourArea)
    if cv2.contourArea(largest) < 0.05 * h * w:  # implausibly small — treat as "no pitch found"
        return np.zeros((h, w), dtype=np.uint8), None, None

    clean_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.drawContours(clean_mask, [largest], -1, 255, thickness=cv2.FILLED)
    bounding_rect = cv2.boundingRect(largest)  # (x, y, w, h)
    polygon_points = largest.reshape(-1, 2)
    return clean_mask, polygon_points, bounding_rect


def point_in_pitch(mask, x, y):
    h, w = mask.shape[:2]
    xi, yi = int(round(x)), int(round(y))
    if 0 <= xi < w and 0 <= yi < h:
        return mask[yi, xi] > 0
    return False


In [ ]:
# ============================================================
# Detection — run the model, filter by confidence, then discard anything whose ground point
# (bottom-center of the box — a better proxy for "where they're standing" than the box center)
# falls outside the pitch mask. This is what removes crowd/stand/ad-board false positives.
# ============================================================
def run_detection(image_pil, pitch_mask, score_threshold):
    img_tensor = TF.to_tensor(image_pil).to(device)
    with torch.no_grad():
        pred = model([img_tensor])[0]

    detections = []
    for box, label, score in zip(pred["boxes"].cpu(), pred["labels"].cpu(), pred["scores"].cpu()):
        if score < score_threshold:
            continue
        x1, y1, x2, y2 = box.tolist()
        ground_x, ground_y = (x1 + x2) / 2, y2

        if pitch_mask is not None and not point_in_pitch(pitch_mask, ground_x, ground_y):
            continue  # likely crowd/stand/ad-board, not on the pitch

        detections.append({
            "class": class_names[label.item() - 1],
            "confidence": float(score),
            "bbox_px": [x1, y1, x2, y2],
            "ground_point_px": [ground_x, ground_y],
        })
    return detections


In [ ]:
# ============================================================
# Team assignment — per image, unsupervised. Crops the TORSO region of each player/goalkeeper
# box (roughly 15%-55% of box height) rather than the whole box, to avoid shorts/socks/ground
# bleeding into the jersey-color estimate. No skin-tone filtering (unreliable across skin tones);
# the torso crop already avoids most skin pixels for typical sports poses.
# ============================================================
def get_jersey_color(image_bgr, bbox_px):
    x1, y1, x2, y2 = [int(round(v)) for v in bbox_px]
    box_h = y2 - y1
    ty1 = y1 + int(0.15 * box_h)
    ty2 = y1 + int(0.55 * box_h)
    ty1, ty2 = max(0, ty1), max(ty1 + 1, ty2)
    x1, x2 = max(0, x1), max(x1 + 1, x2)
    crop = image_bgr[ty1:ty2, x1:x2]
    if crop.size == 0:
        crop = image_bgr[y1:y2, x1:x2]  # fallback to full box if torso crop is degenerate
    if crop.size == 0:
        return np.array([0.0, 0.0, 0.0])
    hsv_crop = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV).reshape(-1, 3).astype(np.float32)
    return hsv_crop.mean(axis=0)  # mean (H, S, V)


def assign_teams(image_bgr, detections, min_players_for_split):
    player_idxs = [i for i, d in enumerate(detections) if d["class"] in ("player", "goalkeeper")]

    for d in detections:
        if d["class"] in ("player", "goalkeeper"):
            d["team"] = "unknown"
        else:
            d["team"] = None  # ball / referee: team assignment doesn't apply

    if len(player_idxs) < min_players_for_split:
        return detections  # too few to cluster meaningfully — leave as "unknown"

    features = np.array([get_jersey_color(image_bgr, detections[i]["bbox_px"]) for i in player_idxs])
    kmeans = KMeans(n_clusters=2, n_init=10, random_state=0).fit(features)

    # Deterministic within THIS image only: lower mean hue -> "A". Not meaningful across images.
    hue_by_cluster = {c: features[kmeans.labels_ == c][:, 0].mean() for c in range(2)}
    ordered = sorted(hue_by_cluster, key=hue_by_cluster.get)
    label_map = {ordered[0]: "A", ordered[1]: "B"}

    for idx, cluster in zip(player_idxs, kmeans.labels_):
        detections[idx]["team"] = label_map[cluster]

    return detections


In [ ]:
# ============================================================
# Position normalization + coverage.
# Position is normalized to the PITCH MASK'S BOUNDING BOX, not a true homography (see notes at
# top) — a reasonable approximation for a relative coverage metric, not a precise measurement.
# ============================================================
def normalize_position(ground_point_px, bounding_rect):
    if bounding_rect is None:
        return None
    x, y, w, h = bounding_rect
    gx, gy = ground_point_px
    nx = np.clip((gx - x) / w, 0.0, 1.0)
    ny = np.clip((gy - y) / h, 0.0, 1.0)
    return [float(nx), float(ny)]


def compute_coverage(detections, bounding_rect):
    if bounding_rect is None:
        return {"team_A": None, "team_B": None, "overall": None}
    _, _, pw, ph = bounding_rect
    pitch_area = pw * ph
    if pitch_area <= 0:
        return {"team_A": None, "team_B": None, "overall": None}

    def hull_area(points):
        if len(points) < 3:
            return None
        pts = np.array(points, dtype=np.float32)
        hull = cv2.convexHull(pts)
        return float(cv2.contourArea(hull))

    team_a_pts = [d["position_norm"] for d in detections if d.get("team") == "A" and d.get("position_norm")]
    team_b_pts = [d["position_norm"] for d in detections if d.get("team") == "B" and d.get("position_norm")]
    all_player_pts = [d["position_norm"] for d in detections
                       if d["class"] in ("player", "goalkeeper") and d.get("position_norm")]

    def pct(points):
        area = hull_area(points)
        return None if area is None else round(100 * area, 2)  # normalized coords -> area already 0-1 scale

    return {"team_A": pct(team_a_pts), "team_B": pct(team_b_pts), "overall": pct(all_player_pts)}


In [ ]:
# ============================================================
# Full per-image pipeline
# ============================================================
def process_image(image_path):
    image_pil = Image.open(image_path).convert("RGB")
    image_bgr = cv2.cvtColor(np.array(image_pil), cv2.COLOR_RGB2BGR)

    pitch_mask, pitch_polygon, bounding_rect = segment_pitch(image_bgr)
    detections = run_detection(image_pil, pitch_mask if pitch_polygon is not None else None,
                                CONFIG["detection_score_threshold"])
    detections = assign_teams(image_bgr, detections, CONFIG["min_players_for_team_split"])

    for d in detections:
        d["position_norm"] = normalize_position(d["ground_point_px"], bounding_rect)

    coverage = compute_coverage(detections, bounding_rect)

    return {
        "image_path": image_path,
        "pitch_detected": pitch_polygon is not None,
        "detections": detections,
        "pitch_coverage": coverage,
    }


In [ ]:
# ============================================================
# Sanity check — ALWAYS look at a few of these before trusting a batch run. Shows the pitch mask
# outline, boxes colored by team, and each team's coverage hull.
# ============================================================
TEAM_COLORS = {"A": "red", "B": "blue", "unknown": "yellow", None: "gray"}

def visualize_result(result, ax):
    image = Image.open(result["image_path"]).convert("RGB")
    ax.imshow(image)

    for d in result["detections"]:
        x1, y1, x2, y2 = d["bbox_px"]
        color = TEAM_COLORS.get(d.get("team"), "gray")
        ax.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                         fill=False, edgecolor=color, linewidth=2))
        label = d["class"] if d["team"] is None else f'{d["class"]} ({d["team"]})'
        ax.text(x1, max(0, y1 - 4), label, color=color, fontsize=8,
                 bbox=dict(facecolor="black", alpha=0.5, pad=0))

    ax.set_title(f'pitch detected: {result["pitch_detected"]} | coverage: {result["pitch_coverage"]}',
                 fontsize=8)
    ax.axis("off")


sample_files = sorted(os.listdir(CONFIG["images_dir"]))[:CONFIG["num_visualization_samples"]]
fig, axes = plt.subplots(1, len(sample_files), figsize=(6 * len(sample_files), 6))
if len(sample_files) == 1:
    axes = [axes]
for ax, fname in zip(axes, sample_files):
    result = process_image(os.path.join(CONFIG["images_dir"], fname))
    visualize_result(result, ax)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Batch run over the full folder — writes one JSON object per line (JSONL) to output_jsonl_path.
# ============================================================
image_exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
image_files = sorted(f for f in os.listdir(CONFIG["images_dir"]) if f.lower().endswith(image_exts))

with open(CONFIG["output_jsonl_path"], "w") as f:
    for fname in tqdm(image_files, desc="Processing images"):
        result = process_image(os.path.join(CONFIG["images_dir"], fname))
        f.write(json.dumps(result) + "\n")

print(f"Wrote {len(image_files)} records to {CONFIG['output_jsonl_path']}")


## Notes / next steps

- **Check the visualization cell's output before trusting the batch run.** If the pitch outline
  looks wrong (e.g. catching a green ad board, or missing most of the actual pitch), the HSV
  auto-calibration failed for that image's lighting/turf — this is the most likely failure mode
  on genuinely new footage.
- **`"team": "unknown"`** shows up whenever an image has fewer than
  `min_players_for_team_split` player/goalkeeper detections — this is intentional, not a bug.
- **Coverage numbers are relative, not physical measurements** — useful for comparing two images
  or two teams within the same image, not for claiming an exact percentage of real pitch area.
- **Revisit the train/val leakage question** from earlier in this conversation if you haven't —
  if frames from the same source video ended up in both your training and validation splits,
  your model's real-world detection quality on unrelated footage (which is what this whole
  pipeline depends on) may be meaningfully worse than your validation mAP suggested.
- If ball recall turns out too low for what you need from this pipeline (checked earlier at
  around 5-6% mAR), consider revisiting the `img_size` / `unfreeze_last_n_blocks` experiments
  before relying on this pipeline's ball-position output specifically.
